# Assignment: Preparing Data for Analysis (Modified Titanic)

**Note:** Each problem starts from the original input DataFrame and is evaluated independently. Do not carry over results or modifications from previous problems. Repeat a previous step only when explicitly instructed.


![](https://github.com/kaopanboonyuen/2110446_DataScience_2021s2/raw/main/%20files/hw.png)

# 1) Load data & review the data

How many rows are there in the "titanic_to_student.csv"?

In [1]:
# Import the libraries
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [13]:
df = pd.read_csv("titanic_to_student.csv")
df.shape[0]

445

# 2) Feature Filtering

2.1 Drop variables with more than 50% missing values.

2.2 Check all columns except `Age` and `Fare` for flat values and drop columns where a flat value is more than 70%.

From 2.1 and 2.2, how many columns remain?

Note: Ensure missing values are considered in the calculation. If using `value_counts()`, include `dropna=False`.


In [14]:
col_to_drop = []
for col in df.columns:
    missing_ratio = df[col].isnull().sum()/len(df)
    if missing_ratio >= 0.5:
        col_to_drop.append(col)
        continue

    if col not in ['Age','Fare']:
        flat_ratio = df[col].value_counts(dropna=False).iloc[0]/len(df)
        if flat_ratio >= 0.7:
            col_to_drop.append(col)
df.drop(columns=col_to_drop)
df.shape[1]

13

# 3) Target Cleaning

Remove all rows with missing values in the target variable `Survived`.

How many rows remain?


In [15]:
# Drop rows with missing values in `Survived`.
df = df.dropna(subset=['Survived'])
df.shape[0]

432

# 4) Outlier Handling

For `Fare`, replace outlier values with the boundary values:

- If value < (Q1 - 1.5IQR), replace it with (Q1 - 1.5IQR).
- If value > (Q3 + 1.5IQR), replace it with (Q3 + 1.5IQR).

What is the mean of `Fare` after replacing the outliers, rounded to 2 decimal places?

Hint: Use `round(_, 2)`.


In [5]:
# Calculate the IQR boundaries and replace the outliers.
q1 = df['Fare'].quantile(q=0.25)
q3 = df['Fare'].quantile(q=0.75)
iqr = q3 - q1

df['Fare'] = df['Fare'].clip(lower=q1-1.5*iqr, upper=q3+1.5*iqr)

round(df['Fare'].mean(),2)

np.float64(26.23)

# 5) Numeric Imputation

Build a scikit-learn numeric preprocessing pipeline.

Separate the target variable `Survived` from the input features. For numeric input columns, build a scikit-learn `Pipeline` using `SimpleImputer(strategy="mean")`. Use `ColumnTransformer` to apply preprocessing to the selected columns.

What is the mean of `Age` after imputing the missing values, rounded to 2 decimal places?

Hint: Use `Pipeline`, `SimpleImputer`, `ColumnTransformer`, and `round(_, 2)`.


In [9]:
# Separate X and y, then build and fit the numeric preprocessing pipeline.
num_cols = df.select_dtypes(include='number').columns.to_list()
cat_cols = [col for col in df.columns if col not in num_cols]

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols)
    ],
    remainder='passthrough'
)

arr_transformed = preprocessor.fit_transform(df)
df_transformed = pd.DataFrame(arr_transformed, columns=num_cols + cat_cols)

round(df_transformed['Age'].mean(),2)

np.float64(29.33)

# 6) Categorical Encoding

Repeat the numeric preprocessing from Problem 5 and separate the target variable `Survived` from the input features.

For `Embarked`, build a categorical `Pipeline` using `SimpleImputer(strategy="most_frequent")` and `OneHotEncoder`. Use `C` as the reference category, handle unknown categories as all zeros, and return dense output.

Combine the numeric and categorical pipelines using `ColumnTransformer` inside a `Pipeline`.

What is the mean of `Embarked_Q`, rounded to 2 decimal places?

Hint: Check the `drop`, `handle_unknown`, and `sparse_output` options, and use `round(_, 2)`.


In [10]:
# Repeat the numeric preprocessing, add the `Embarked` branch, and fit the combined pipeline.

num_cols = df.select_dtypes(include='number').columns.drop('Survived', errors='ignore').tolist()
cat_cols = ['Embarked']

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop=['C'],handle_unknown='ignore',sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ],
    remainder='drop'
)

arr_transformed = preprocessor.fit_transform(df)
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols)
all_feature_names = list(cat_feature_names) + num_cols

df_transformed = pd.DataFrame(arr_transformed, columns=all_feature_names)

round(df_transformed['Embarked_Q'].mean(),2)

np.float64(0.06)

# 7) Train-Test Splitting

Repeat the target cleaning from Problem 3.

Separate the input features (`X`) and target (`y`). Split the data into training and test sets using a 70:30 ratio, `random_state=123`, and `stratify=y`.

What is the proportion of survivors in the training data, rounded to 2 decimal places?

Hint: Use `dropna()`, `round(_, 2)`, and `train_test_split()` from `sklearn.model_selection`.


In [19]:
# Repeat the target cleaning, separate X and y, and perform the stratified split.
from sklearn.model_selection import train_test_split

x = df.drop(columns=['Survived'])
y = df['Survived']

x_train, x_test, y_train, y_test = train_test_split(
    x, 
    y, 
    test_size=0.3,
    random_state=123,
    stratify=y
)

round(y_train.mean(),2)

np.float64(0.42)